# UPF Literature — Author Co-authorship Network

Builds and analyses a co-authorship network from the edge list produced by
`upf_bibliometrics.py`.  
**Pre-requisite:** run `python upf_bibliometrics.py` (or `--dry-run` for a
quick test) so that `output/coauthorship_edges.csv` exists.

**Install once** (if not already present):
```bash
pip install networkx
```

In [ ]:
import collections
import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import networkx as nx
import pandas as pd

warnings.filterwarnings('ignore')

# ── Paths (relative to notebook location) ─────────────────────────────────────
EDGES_CSV   = os.path.join('..', 'output', 'coauthorship_edges.csv')
AUTHORS_CSV = os.path.join('..', 'output', 'papers_by_author.csv')
OUTPUT_DIR  = os.path.join('..', 'output')

# ── Tunable parameters ─────────────────────────────────────────────────────────
MIN_PAPERS      = 3    # keep only authors with >= this many papers in the corpus
MIN_EDGE_WEIGHT = 2    # keep only co-authorship pairs that share >= this many papers
TOP_N_LABELS    = 30   # label the top-N highest-degree nodes in the graph plot
LAYOUT_SEED     = 42

print('NetworkX version:', nx.__version__)

## 1. Load data

In [ ]:
edges_df   = pd.read_csv(EDGES_CSV)
authors_df = pd.read_csv(AUTHORS_CSV)

print(f'Edge rows   : {len(edges_df):,}')
print(f'Author rows : {len(authors_df):,}')
edges_df.head(3)

## 2. Build the full graph, then filter

We keep only nodes (authors) that appear in at least `MIN_PAPERS` papers **and**
edges (co-authorship pairs) with at least `MIN_EDGE_WEIGHT` shared papers.
This removes noise from one-off collaborations and focuses the network on
the productive core of the field.

In [ ]:
# Node attribute lookup: author_id → {name, institution, country, papers}
node_attrs = (
    authors_df
    .set_index('author_id')[['author_name', 'institution', 'country', 'papers']]
    .to_dict(orient='index')
)

# Productive-author set
core_authors = {
    aid for aid, attr in node_attrs.items()
    if attr['papers'] >= MIN_PAPERS
}
print(f'Authors with >= {MIN_PAPERS} papers : {len(core_authors):,}')

# Build graph
G = nx.Graph()

for _, row in edges_df.iterrows():
    a1, a2, w = row['author1_id'], row['author2_id'], row['shared_papers']
    if a1 not in core_authors or a2 not in core_authors:
        continue
    if w < MIN_EDGE_WEIGHT:
        continue
    if G.has_edge(a1, a2):
        G[a1][a2]['weight'] += w
    else:
        G.add_edge(a1, a2, weight=w)

# Attach node attributes
for node in G.nodes():
    attrs = node_attrs.get(node, {})
    G.nodes[node]['name']        = attrs.get('author_name', node)
    G.nodes[node]['institution'] = attrs.get('institution', '')
    G.nodes[node]['country']     = attrs.get('country', '')
    G.nodes[node]['papers']      = attrs.get('papers', 0)

# Focus on the largest connected component
lcc_nodes = max(nx.connected_components(G), key=len)
G_lcc     = G.subgraph(lcc_nodes).copy()

print(f'Full graph     : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'Largest component : {G_lcc.number_of_nodes():,} nodes, {G_lcc.number_of_edges():,} edges')

## 3. Global network statistics

In [ ]:
n = G_lcc.number_of_nodes()
m = G_lcc.number_of_edges()
density   = nx.density(G_lcc)
avg_deg   = 2 * m / n if n else 0
avg_clust = nx.average_clustering(G_lcc, weight='weight')
diameter  = nx.diameter(G_lcc) if n < 5_000 else 'skipped (graph too large)'
avg_path  = nx.average_shortest_path_length(G_lcc) if n < 5_000 else 'skipped'
components_full = nx.number_connected_components(G)

print('── Network statistics (largest component) ──────────────')
print(f'  Nodes                 : {n:,}')
print(f'  Edges                 : {m:,}')
print(f'  Density               : {density:.4f}')
print(f'  Average degree        : {avg_deg:.2f}')
print(f'  Average clustering    : {avg_clust:.4f}')
print(f'  Diameter              : {diameter}')
print(f'  Avg shortest path     : {avg_path}')
print(f'  Components (full graph): {components_full:,}')

## 4. Centrality measures

In [ ]:
degree_cent     = nx.degree_centrality(G_lcc)
betweenness     = nx.betweenness_centrality(G_lcc, weight='weight', normalized=True)
pagerank        = nx.pagerank(G_lcc, weight='weight')
clustering      = nx.clustering(G_lcc, weight='weight')

centrality_df = pd.DataFrame({
    'author_id'   : list(G_lcc.nodes()),
    'name'        : [G_lcc.nodes[n]['name']        for n in G_lcc.nodes()],
    'institution' : [G_lcc.nodes[n]['institution'] for n in G_lcc.nodes()],
    'country'     : [G_lcc.nodes[n]['country']     for n in G_lcc.nodes()],
    'papers'      : [G_lcc.nodes[n]['papers']      for n in G_lcc.nodes()],
    'degree'      : [G_lcc.degree(n)               for n in G_lcc.nodes()],
    'degree_centrality'  : [degree_cent[n]   for n in G_lcc.nodes()],
    'betweenness'        : [betweenness[n]   for n in G_lcc.nodes()],
    'pagerank'           : [pagerank[n]      for n in G_lcc.nodes()],
    'clustering'         : [clustering[n]   for n in G_lcc.nodes()],
})

centrality_df.sort_values('betweenness', ascending=False, inplace=True)
centrality_df.reset_index(drop=True, inplace=True)

# Save
out_path = os.path.join(OUTPUT_DIR, 'author_centrality.csv')
centrality_df.to_csv(out_path, index=False)
print(f'Saved → {out_path}')

print('\nTop 15 by betweenness centrality (bridges / gatekeepers):')
centrality_df[['name', 'institution', 'country', 'papers', 'degree', 'betweenness', 'pagerank']].head(15)

In [ ]:
print('Top 15 by degree (most direct collaborators):')
centrality_df.sort_values('degree', ascending=False).head(15)[
    ['name', 'institution', 'country', 'papers', 'degree', 'betweenness']
]

## 5. Community detection

Uses the Louvain algorithm (built into NetworkX ≥ 2.7) to partition the
network into research communities.

In [ ]:
communities = nx.community.louvain_communities(G_lcc, weight='weight', seed=LAYOUT_SEED)
communities = sorted(communities, key=len, reverse=True)

print(f'Number of communities detected: {len(communities)}')
print(f'Sizes of top-10 communities   : {[len(c) for c in communities[:10]]}')

# Assign community labels to nodes
node_community = {}
for idx, community in enumerate(communities):
    for node in community:
        node_community[node] = idx

nx.set_node_attributes(G_lcc, node_community, 'community')

# Add to centrality table
centrality_df['community'] = centrality_df['author_id'].map(node_community)

# Per-community summary
comm_summary = (
    centrality_df
    .groupby('community')
    .agg(
        size=('name', 'count'),
        total_papers=('papers', 'sum'),
        top_country=('country', lambda x: x.value_counts().index[0] if len(x) else ''),
        top_institution=('institution', lambda x: x.value_counts().index[0] if len(x) else ''),
    )
    .sort_values('size', ascending=False)
    .head(10)
)
print('\nTop-10 communities:')
comm_summary

## 6. Visualisations

### 6a. Degree distribution

In [ ]:
degrees = [d for _, d in G_lcc.degree()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(degrees, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Degree')
axes[0].set_ylabel('Count')
axes[0].set_title('Degree distribution (linear scale)')

# Log-log plot to check for power-law behaviour
freq = collections.Counter(degrees)
xs, ys = zip(*sorted(freq.items()))
axes[1].loglog(xs, ys, 'o', markersize=4, color='steelblue')
axes[1].set_xlabel('Degree (log)')
axes[1].set_ylabel('Count (log)')
axes[1].set_title('Degree distribution (log-log)')

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'degree_distribution.png'), dpi=150)
plt.show()
print(f'Mean degree: {sum(degrees)/len(degrees):.2f}  Max: {max(degrees)}')

### 6b. Co-authorship network (largest component, coloured by community)

In [ ]:
# For legibility, cap the plotted graph at 500 nodes (highest-degree subset)
PLOT_CAP = 500
if G_lcc.number_of_nodes() > PLOT_CAP:
    top_nodes = sorted(G_lcc.nodes(), key=lambda n: G_lcc.degree(n), reverse=True)[:PLOT_CAP]
    G_plot = G_lcc.subgraph(top_nodes).copy()
    print(f'Plotting top-{PLOT_CAP} nodes by degree (graph has {G_lcc.number_of_nodes():,} total)')
else:
    G_plot = G_lcc

pos = nx.spring_layout(G_plot, weight='weight', seed=LAYOUT_SEED, k=0.8)

n_communities = max(node_community.values()) + 1 if node_community else 1
cmap = cm.get_cmap('tab20', min(n_communities, 20))

node_colors = [
    cmap(G_plot.nodes[n].get('community', 0) % 20)
    for n in G_plot.nodes()
]
node_sizes = [
    20 + 15 * G_plot.degree(n)
    for n in G_plot.nodes()
]
edge_widths = [
    0.3 + 0.5 * G_plot[u][v].get('weight', 1)
    for u, v in G_plot.edges()
]

fig, ax = plt.subplots(figsize=(16, 14))

nx.draw_networkx_edges(
    G_plot, pos, ax=ax,
    width=edge_widths, alpha=0.25, edge_color='#555555'
)
nx.draw_networkx_nodes(
    G_plot, pos, ax=ax,
    node_color=node_colors, node_size=node_sizes, alpha=0.85
)

# Label top-N nodes
top_label_nodes = sorted(G_plot.nodes(), key=lambda n: G_plot.degree(n), reverse=True)[:TOP_N_LABELS]
labels = {n: G_plot.nodes[n].get('name', n).split()[-1] for n in top_label_nodes}  # last name only
nx.draw_networkx_labels(
    G_plot, pos, labels=labels, ax=ax,
    font_size=7, font_color='black'
)

ax.set_title(
    f'UPF Co-authorship Network  |  '
    f'{G_plot.number_of_nodes()} authors, {G_plot.number_of_edges()} edges  |  '
    f'node colour = community',
    fontsize=12
)
ax.axis('off')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'author_network.png'), dpi=150)
plt.show()

### 6c. Betweenness vs degree scatter — identifying bridges

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sc = ax.scatter(
    centrality_df['degree'],
    centrality_df['betweenness'],
    c=centrality_df['community'].fillna(0),
    cmap='tab20',
    s=centrality_df['papers'] * 3,
    alpha=0.6,
    linewidths=0,
)

# Annotate outliers: high betweenness relative to degree
bridge_thresh = centrality_df['betweenness'].quantile(0.97)
for _, row in centrality_df[centrality_df['betweenness'] >= bridge_thresh].iterrows():
    ax.annotate(
        row['name'].split()[-1],
        (row['degree'], row['betweenness']),
        fontsize=7, xytext=(4, 2), textcoords='offset points'
    )

ax.set_xlabel('Degree (number of co-authors)')
ax.set_ylabel('Betweenness centrality')
ax.set_title('Degree vs Betweenness  |  point size ∝ papers  |  colour = community')
plt.colorbar(sc, label='Community', ax=ax)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'betweenness_vs_degree.png'), dpi=150)
plt.show()

### 6d. Country-level collaboration heatmap

Counts co-authored papers between pairs of countries.

In [ ]:
country_edges = collections.Counter()
for u, v, data in G_lcc.edges(data=True):
    c1 = G_lcc.nodes[u].get('country', '')
    c2 = G_lcc.nodes[v].get('country', '')
    if c1 and c2 and c1 != c2:
        pair = tuple(sorted([c1, c2]))
        country_edges[pair] += data.get('weight', 1)

top_pairs = country_edges.most_common(20)
countries_involved = sorted({c for pair in top_pairs for c in pair})

# Build matrix
matrix = pd.DataFrame(0, index=countries_involved, columns=countries_involved)
for (c1, c2), w in top_pairs:
    if c1 in matrix.index and c2 in matrix.columns:
        matrix.loc[c1, c2] = w
        matrix.loc[c2, c1] = w

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(matrix.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(countries_involved)))
ax.set_yticks(range(len(countries_involved)))
ax.set_xticklabels(countries_involved, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(countries_involved, fontsize=9)
plt.colorbar(im, ax=ax, label='Co-authored papers')
ax.set_title('Cross-country co-authorship (top-20 pairs)')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'country_collaboration_heatmap.png'), dpi=150)
plt.show()

## 7. Save enriched centrality table

In [ ]:
out_path = os.path.join(OUTPUT_DIR, 'author_centrality.csv')
cols = ['author_id', 'name', 'institution', 'country', 'community',
        'papers', 'degree', 'degree_centrality', 'betweenness', 'pagerank', 'clustering']
centrality_df[cols].sort_values('betweenness', ascending=False).to_csv(out_path, index=False)
print(f'Saved enriched centrality table → {out_path}')
centrality_df[cols].head(10)